# 02 Data Cleaning

Removes duplicates, imputes missing numeric columns with medians, imputes categorical columns with modes, and saves cleaned files to `ai/preprocessing/cleaned`.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')

CLEANED_DIR = PREPROCESSING_DIR / 'cleaned'
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

def clean_dataframe(df):
    before = {'rows': len(df), 'columns': df.shape[1], 'duplicates': int(df.duplicated().sum()), 'missing': int(df.isna().sum().sum())}
    cleaned = df.drop_duplicates().copy()
    numeric_cols = cleaned.select_dtypes(include='number').columns
    categorical_cols = cleaned.columns.difference(numeric_cols)
    for col in numeric_cols:
        cleaned[col] = cleaned[col].fillna(cleaned[col].median())
    for col in categorical_cols:
        mode = cleaned[col].mode(dropna=True)
        fill = mode.iloc[0] if not mode.empty else 'Unknown'
        cleaned[col] = cleaned[col].fillna(fill)
    after = {'rows': len(cleaned), 'columns': cleaned.shape[1], 'duplicates': int(cleaned.duplicated().sum()), 'missing': int(cleaned.isna().sum().sum())}
    return cleaned, before, after


In [ ]:
for path in discover_data_files():
    display_heading(path.relative_to(DATA_DIR).as_posix())
    try:
        df = read_dataset(path)
        cleaned, before, after = clean_dataframe(df)
        output = CLEANED_DIR / safe_name(path)
        sep = '\t' if output.suffix.lower() == '.tsv' else ','
        cleaned.to_csv(output, index=False, sep=sep)
        display(pd.DataFrame([before, after], index=['before', 'after']))
        print('Saved:', output.relative_to(PROJECT_ROOT))
    except Exception as exc:
        print(f'Cleaning failed for {path.name}: {exc}')
